install dependencies

imports and models

In [ ]:
# Install dependencies
!pip install -q torch pandas openpyxl esm huggingface_hub

In [ ]:
import re
import math
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any
# Optional: only needed if the HF repo is private
#from huggingface_hub import login
#login()
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [ ]:
# =========================================================
# 1) Parsing cyclization patterns
# =========================================================
_SIMPLE_PAIR_RE = re.compile(r"^\s*(\d+)\s*-\s*(\d+)\s*$")

def clean_sequence(seq: str) -> str:
    return str(seq).strip().replace(" ", "").replace("\n", "").upper()

def parse_simple_cyclization_pairs(cyclization_pattern: Any) -> List[Tuple[int, int]]:
    """
    Parse strings like:
        "1-8"
        "1-8, 2-40"
    """
    if cyclization_pattern is None:
        return []

    s = str(cyclization_pattern).strip()
    if not s or s.lower() in {"none", "linear", "nan"}:
        return []

    pairs = []
    for token in s.split(","):
        token = token.strip()
        if not token:
            continue
        m = _SIMPLE_PAIR_RE.match(token)
        if not m:
            raise ValueError(
                f"Invalid cyclization token '{token}'. Expected format like '1-8' or '1-8, 2-40'."
            )
        i, j = int(m.group(1)), int(m.group(2))
        if i == j:
            raise ValueError(f"Invalid cyclization pair '{token}': residues cannot be identical.")
        pairs.append((i, j))
    return pairs


# =========================================================
# 2) Graph + shortest path matrix
# =========================================================
def build_adjacency(n: int, pairs_1based: List[Tuple[int, int]]) -> List[List[int]]:
    adj = [[] for _ in range(n)]

    # backbone edges
    for i in range(n - 1):
        adj[i].append(i + 1)
        adj[i + 1].append(i)

    # cyclization edges
    for i, j in pairs_1based:
        if not (1 <= i <= n and 1 <= j <= n):
            raise ValueError(f"Cyclization pair ({i}, {j}) is outside sequence length {n}.")
        a, b = i - 1, j - 1
        if a != b:
            adj[a].append(b)
            adj[b].append(a)

    return [sorted(set(nei)) for nei in adj]


def shortest_path_matrix(adj: List[List[int]]) -> np.ndarray:
    from collections import deque

    n = len(adj)
    D = np.full((n, n), fill_value=10**9, dtype=np.int32)

    for s in range(n):
        D[s, s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            du = D[s, u]
            for v in adj[u]:
                if D[s, v] > du + 1:
                    D[s, v] = du + 1
                    q.append(v)
    return D


# =========================================================
# 3) ESM-C loading + embeddings
# =========================================================
def load_esmc(model_name: str = "esmc_300m",
              device: str = "cuda" if torch.cuda.is_available() else "cpu"):
    from esm.models.esmc import ESMC
    model = ESMC.from_pretrained(model_name).to(device)
    model.eval()
    return model


@torch.no_grad()
def esmc_token_embeddings(model, sequence: str, device: str) -> torch.Tensor:
    from esm.sdk.api import ESMProtein, LogitsConfig

    sequence = clean_sequence(sequence)
    L = len(sequence)

    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein).to(device)
    out = model.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
    emb = out.embeddings

    if isinstance(emb, np.ndarray):
        emb = torch.from_numpy(emb)

    emb = emb.to(device).detach()

    if emb.dim() == 3:
        if emb.shape[0] != 1:
            raise RuntimeError(f"Expected batch=1, got shape {tuple(emb.shape)}")
        emb = emb.squeeze(0)

    if emb.dim() != 2:
        raise RuntimeError(f"Expected 2D embeddings, got {tuple(emb.shape)}")

    T = emb.shape[0]

    if T == L:
        return emb
    if T == L + 2:
        return emb[1:-1, :]
    if T == L + 1:
        return emb[1:, :]
    if T > L + 2:
        start = (T - L) // 2
        return emb[start:start + L, :]

    raise RuntimeError(f"Cannot align embeddings: tokens={T}, seq_len={L}")


# =========================================================
# 4) Model definition
# =========================================================
class RelativeBiasSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dmax: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

        self.rel_bias = nn.Embedding(dmax + 1, n_heads)

        self.ln = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, D: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        h = self.ln(x)

        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_head)

        rb = self.rel_bias(D).permute(0, 3, 1, 2).contiguous()
        scores = scores + rb

        key_mask = (~mask).view(B, 1, 1, L)
        scores = scores.masked_fill(key_mask, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.drop(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        out = self.out(out)
        out = self.drop(out)

        x = x + out
        x = x + self.ff(x)
        return x


class CycOffsetRegressor(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dmax: int,
                 n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            RelativeBiasSelfAttention(d_model=d_model, n_heads=n_heads, dmax=dmax, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x: torch.Tensor, D: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        for blk in self.blocks:
            x = blk(x, D, mask)

        x = self.final_ln(x)

        m = mask.unsqueeze(-1)
        x_masked = x * m
        lengths = m.sum(dim=1).clamp(min=1.0)

        mean_pool = x_masked.sum(dim=1) / lengths

        x_for_max = x.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        max_pool = torch.max(x_for_max, dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        z = torch.cat([mean_pool, max_pool], dim=-1)
        return self.mlp(z)

In [ ]:
@dataclass
class PredictorConfig:
    checkpoint_path: str
    model_name: str = "esmc_300m"
    dmax: int = 10
    n_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class Stop2MeltPredictor:
    def __init__(self, config: PredictorConfig):
        self.config = config
        self.device = config.device

        print(f"[INFO] device: {self.device}")
        print(f"[INFO] loading ESM-C model: {config.model_name}")
        self.esmc = load_esmc(model_name=config.model_name, device=self.device)

        dummy_seq = "ACDEFGHIK"
        dummy_emb = esmc_token_embeddings(self.esmc, dummy_seq, self.device)
        d_model = dummy_emb.shape[1]
        print(f"[INFO] detected embedding dim: {d_model}")

        self.model = CycOffsetRegressor(
            d_model=d_model,
            n_heads=config.n_heads,
            dmax=config.dmax,
            n_layers=config.n_layers,
            dropout=config.dropout,
        ).to(self.device)

        print(f"[INFO] loading checkpoint: {config.checkpoint_path}")
        state = torch.load(config.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(state)
        self.model.eval()
        print("[INFO] predictor ready.")

    @torch.no_grad()
    def predict_one(self, sequence: str, cyclization_pattern: Optional[str] = None) -> Dict[str, Any]:
        sequence = clean_sequence(sequence)
        if len(sequence) == 0:
            raise ValueError("Sequence is empty.")

        pairs = parse_simple_cyclization_pairs(cyclization_pattern)
        adj = build_adjacency(len(sequence), pairs)
        D = shortest_path_matrix(adj)
        D = np.clip(D, 0, self.config.dmax).astype(np.int64)

        tok = esmc_token_embeddings(self.esmc, sequence, self.device)

        L = tok.shape[0]
        X = tok.unsqueeze(0).to(dtype=torch.float32, device=self.device)
        D_tensor = torch.from_numpy(D).unsqueeze(0).to(self.device)
        mask = torch.ones((1, L), dtype=torch.bool, device=self.device)

        pred = self.model(X, D_tensor, mask).squeeze().item()

        return {
            "Sequence": sequence,
            "CyclizationPattern": cyclization_pattern if cyclization_pattern is not None else "",
            "Pred_Stop2Melt": float(pred),
        }

    @torch.no_grad()
    def predict_batch(self, df: pd.DataFrame,
                      seq_col: str = "Sequence",
                      cycl_col: str = "CyclizationPattern") -> pd.DataFrame:
        if seq_col not in df.columns:
            raise ValueError(f"Missing required column: {seq_col}")

        if cycl_col not in df.columns:
            df = df.copy()
            df[cycl_col] = ""

        results = []
        for _, row in df.iterrows():
            seq = row[seq_col]
            cyc = row[cycl_col]
            try:
                out = self.predict_one(seq, cyc)
                results.append(out)
            except Exception as e:
                results.append({
                    "Sequence": seq,
                    "CyclizationPattern": cyc,
                    "Pred_Stop2Melt": np.nan,
                    "Error": str(e),
                })

        return pd.DataFrame(results)

In [ ]:
# Download checkpoint from Hugging Face
from huggingface_hub import hf_hub_download

repo_id = "KarunaAnna/STop2Melt"   # huggingface
filename = "cycoffset_esmc_cymelt.pt"       # pt file for trained model

checkpoint_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="model",   # change to "dataset" only if the file lives in a dataset repo
    # revision="main",    # optional: branch, tag, or full commit hash
)

print("Checkpoint path:", checkpoint_path)

Checkpoint path: /root/.cache/huggingface/hub/models--KarunaAnna--STop2Melt/snapshots/668eb64fe926d3c8c9b13ea6c17cb51fb628487f/cycoffset_esmc_cymelt.pt


In [ ]:
config = PredictorConfig(
    checkpoint_path=checkpoint_path,
    model_name="esmc_300m",
    dmax=10,
    n_layers=2,
    n_heads=8,
    dropout=0.1,
)

predictor = Stop2MeltPredictor(config)

[INFO] device: cpu
[INFO] loading ESM-C model: esmc_300m


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/weights/esmc_300m_2024_12_v0.pth:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

[INFO] detected embedding dim: 960
[INFO] loading checkpoint: /root/.cache/huggingface/hub/models--KarunaAnna--STop2Melt/snapshots/668eb64fe926d3c8c9b13ea6c17cb51fb628487f/cycoffset_esmc_cymelt.pt
[INFO] predictor ready.


In [ ]:
from google.colab import files

In [ ]:
def option_single_sequence(predictor):
    seq = input("Enter one peptide sequence: ").strip()
    cyc = input("Enter cyclization pattern (e.g. 1-8 or 1-20, 3-15). Leave blank if none: ").strip()

    result = predictor.predict_one(seq, cyc)
    df = pd.DataFrame([result])
    print("\nPrediction:")
    display(df)
    return df


def option_multiple_sequences(predictor):
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed): ").strip()
        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df_input = pd.DataFrame(rows)
    df_pred = predictor.predict_batch(df_input)
    print("\nPredictions:")
    display(df_pred)
    return df_pred


def option_upload_csv_or_excel(predictor):
    print("Upload a CSV or Excel file containing sequence and cyclization columns.")
    uploaded = files.upload()

    input_file = list(uploaded.keys())[0]

    if input_file.lower().endswith(".csv"):
        df_input = pd.read_csv(input_file)
    elif input_file.lower().endswith((".xlsx", ".xls")):
        df_input = pd.read_excel(input_file)
    else:
        raise ValueError("Please upload a CSV or Excel file.")

    print("Detected columns:", list(df_input.columns))
    seq_col = input("Enter sequence column name [default: Sequence]: ").strip() or "Sequence"
    cycl_col = input("Enter cyclization column name [default: CyclizationPattern]: ").strip() or "CyclizationPattern"

    df_pred = predictor.predict_batch(df_input, seq_col=seq_col, cycl_col=cycl_col)
    print("\nPredictions:")
    display(df_pred.head())
    return df_pred


def run_user_menu(predictor):
    print("\nChoose input mode:")
    print("1 = One sequence")
    print("2 = Multiple sequences manually")
    print("3 = Upload CSV/Excel file")

    choice = input("Enter 1, 2, or 3: ").strip()

    if choice == "1":
        return option_single_sequence(predictor)
    elif choice == "2":
        return option_multiple_sequences(predictor)
    elif choice == "3":
        return option_upload_csv_or_excel(predictor)
    else:
        raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

In [ ]:
df_results = run_user_menu(predictor)


Choose input mode:
1 = One sequence
2 = Multiple sequences manually
3 = Upload CSV/Excel file
Enter 1, 2, or 3: 1
Enter one peptide sequence: AKLAFKKLFQLICCCFK
Enter cyclization pattern (e.g. 1-8 or 1-20, 3-15). Leave blank if none: 

Prediction:


,Sequence,CyclizationPattern,Pred_Stop2Melt
0,AKLAFKKLFQLICCCFK,,324.744354
